#### Accounts Table

In [1]:
def generate_accounts():
    accounts = []

    industries = ["Fintech", "HealthTech", "EdTech", "E-commerce", "SaaS"]
    company_sizes = ["Small", "Medium", "Large"]

    for i in range(1, NUM_ACCOUNTS + 1):
        account = {
            "account_id": i,
            "account_name": fake.company(),
            "industry": random.choice(industries),
            "company_size": random.choice(company_sizes),
            "created_at": fake.date_between(start_date="-2y", end_date="today")
        }
        accounts.append(account)

    return pd.DataFrame(accounts)

#### Customers Table

In [2]:
def generate_customers(accounts_df):
    customers = []
    customer_id = 1

    for _, row in accounts_df.iterrows():
        account_id = row["account_id"]
        company_size = row["company_size"]

        # determine number of users based on company size
        if company_size == "Small":
            num_customers = random.randint(1, 3)
        elif company_size == "Medium":
            num_customers = random.randint(3, 6)
        else:
            num_customers = random.randint(6, 12)

        for _ in range(num_customers):
            customer = {
                "customer_id": customer_id,
                "account_id": account_id,
                "email": fake.email(),
                "role": random.choice(["admin", "user"]),
                "created_at": fake.date_between(start_date="-2y", end_date="today")
            }
            customers.append(customer)
            customer_id += 1

    return pd.DataFrame(customers)

#### Sales Rep

In [3]:
NUM_REPS = 30  # total sales reps in your simulated company

def generate_sales_reps():
    reps = []
    for i in range(1, NUM_REPS + 1):
        reps.append({
            "rep_id": i,
            "rep_name": fake.name(),
            "email": fake.email(),
            "region": random.choice(["EMEA", "APAC", "AMERICAS"])
        })
    return pd.DataFrame(reps)

#### Deals Stages

In [4]:
def generate_deal_stages():
    stages = [
        {"stage_id": 1, "stage_name": "Lead", "win_probability": 0.1},
        {"stage_id": 2, "stage_name": "Qualified", "win_probability": 0.3},
        {"stage_id": 3, "stage_name": "Proposal", "win_probability": 0.6},
        {"stage_id": 4, "stage_name": "Negotiation", "win_probability": 0.8},
        {"stage_id": 5, "stage_name": "Closed Won", "win_probability": 1.0},
        {"stage_id": 6, "stage_name": "Closed Lost", "win_probability": 0.0}
    ]
    return pd.DataFrame(stages)

#### Deals Table

In [5]:
def generate_deals(accounts_df):
    deals = []
    deal_id = 1

    for _, row in accounts_df.iterrows():
        account_id = row["account_id"]

        # each account has 1–4 deals
        num_deals = random.randint(1, 4)

        for _ in range(num_deals):
            created_at = fake.date_between(start_date="-1y", end_date="today")

            # assign outcome
            outcome = random.choices(
                ["open", "won", "lost"],
                weights=[0.55, 0.3, 0.15]
            )[0]

            # deal value (won deals slightly higher)
            if outcome == "won":
                deal_value = random.randint(5000, 20000)
            elif outcome == "lost":
                deal_value = random.randint(1000, 10000)
            else:
                deal_value = random.randint(2000, 15000)

            # stage logic
            if outcome == "won":
                stage_id = 5
                closed_at = created_at + timedelta(days=random.randint(10, 60))
            elif outcome == "lost":
                stage_id = 6
                closed_at = created_at + timedelta(days=random.randint(5, 40))
            else:
                stage_id = random.randint(1, 4)
                closed_at = None

            # expected close date (for forecasting)
            expected_close_date = created_at + timedelta(days=random.randint(15, 90))

            # JSONB attributes
            attributes = {
                "lead_source": random.choice(["LinkedIn", "Website", "Referral"]),
                "priority": random.choice(["low", "medium", "high"]),
                "industry": row["industry"]
            }
            # assign a random sales rep to the deal
            rep_id = random.randint(1, NUM_REPS)  
            deal = {
                "deal_id": deal_id,
                "account_id": account_id,
                "deal_value": deal_value,
                "stage_id": stage_id,
                "status": outcome,
                "created_at": created_at,
                "expected_close_date": expected_close_date,
                "closed_at": closed_at,
                "rep_id": rep_id,
                "attributes": json.dumps(attributes)
            }

            deals.append(deal)
            deal_id += 1

    return pd.DataFrame(deals)

#### Activities Table

In [6]:
def generate_activities(deals_df):
    activities = []
    activity_id = 1

    for _, deal in deals_df.iterrows():
        deal_id = deal["deal_id"]
        status = deal["status"]

        # decide number of activities
        if status == "won":
            num_activities = random.randint(3, 7)
        elif status == "lost":
            num_activities = random.randint(1, 3)
        else:  # open
            num_activities = random.randint(1, 5)

        last_activity_date = deal["created_at"]

        for _ in range(num_activities):
            # increment activity date
            activity_date = last_activity_date + timedelta(days=random.randint(1, 10))
            last_activity_date = activity_date

            activity_type = random.choice(["call", "email", "demo", "meeting"])
            details = {
                "call_duration": random.randint(60, 600) if activity_type == "call" else None,
                "outcome": random.choice(["positive", "neutral", "negative"]),
                "notes": fake.sentence()
            }

            activity = {
                "activity_id": activity_id,
                "deal_id": deal_id,
                "activity_type": activity_type,
                "activity_date": activity_date,
                "details": json.dumps(details)
            }

            activities.append(activity)
            activity_id += 1

    return pd.DataFrame(activities)

#### Subscriptions Table

In [7]:
def generate_subscriptions(accounts_df):
    subscriptions = []
    subscription_id = 1

    plans = [("Basic", 50), ("Pro", 100), ("Premium", 200)]

    for _, row in accounts_df.iterrows():
        account_id = row["account_id"]
        company_size = row["company_size"]

        # determine number of subscriptions
        if company_size == "Small":
            num_subs = 1
        elif company_size == "Medium":
            num_subs = random.randint(1, 2)
        else:  # Large
            num_subs = random.randint(2, 3)

        for _ in range(num_subs):
            start_date = fake.date_between(start_date="-18M", end_date="-6M")
            plan_name, price = random.choice(plans)

            # determine status
            status_roll = random.random()
            if status_roll < 0.2:  # churned
                status = "cancelled"
                end_date = start_date + timedelta(days=random.randint(30, 365))
            else:
                status = "active"
                end_date = None

            # some upgrades
            if status == "active" and random.random() < 0.15:
                plan_name = "Pro" if plan_name == "Basic" else "Premium"

            subscription = {
                "subscription_id": subscription_id,
                "account_id": account_id,
                "plan_name": plan_name,
                "start_date": start_date,
                "end_date": end_date,
                "status": status
            }
            subscriptions.append(subscription)
            subscription_id += 1

    return pd.DataFrame(subscriptions)

#### Transactions Table

In [8]:
def generate_transactions(subscriptions_df):
    transactions = []
    transaction_id = 1

    for _, sub in subscriptions_df.iterrows():
        sub_id = sub["subscription_id"]
        plan_name = sub["plan_name"]
        amount = 50 if plan_name == "Basic" else 100 if plan_name == "Pro" else 200

        start_date = sub["start_date"]
        end_date = sub["end_date"] or datetime.today().date()

        current_date = start_date
        while current_date <= end_date:
            # simulate occasional failed payment (5% chance)
            if random.random() < 0.05:
                current_date += timedelta(days=30)
                continue

            transaction = {
                "transaction_id": transaction_id,
                "subscription_id": sub_id,
                "amount": amount,
                "transaction_date": current_date,
                "type": "payment"
            }
            transactions.append(transaction)
            transaction_id += 1

            # move to next month
            current_date += timedelta(days=30)

    return pd.DataFrame(transactions)

#### Usage Logs Table

In [9]:
def generate_usage_logs(customers_df):
    usage_logs = []
    usage_id = 1

    features = ["dashboard", "report", "export", "settings", "integration"]
    devices = ["web", "mobile"]

    for _, customer in customers_df.iterrows():
        customer_id = customer["customer_id"]

        # assign user behavior type
        behavior = random.choices(
            ["active", "moderate", "churn_risk"],
            weights=[0.6, 0.25, 0.15]
        )[0]

        start_date = datetime.today() - timedelta(days=365)
        
        for day_offset in range(365):
            current_date = start_date + timedelta(days=day_offset)

            # determine activity frequency
            if behavior == "active":
                num_events = random.randint(1, 3)

            elif behavior == "moderate":
                num_events = random.choice([0, 1, 2])

            else:  # churn_risk
                # decline over time
                decline_factor = max(0, 1 - (day_offset / 365))
                num_events = 1 if random.random() < decline_factor else 0

            for _ in range(num_events):
                metadata = {
                    "feature": random.choice(features),
                    "device": random.choice(devices),
                    "duration_seconds": random.randint(30, 600),
                    "location": fake.country()
                }

                usage_log = {
                    "usage_id": usage_id,
                    "customer_id": customer_id,
                    "activity_type": "feature_use",
                    "activity_date": current_date,
                    "metadata": json.dumps(metadata)
                }

                usage_logs.append(usage_log)
                usage_id += 1

    return pd.DataFrame(usage_logs)

#### Churn Table

In [10]:
def generate_churn_events(accounts_df, customers_df, subscriptions_df, usage_logs_df):
    churn_events = []
    churn_id = 1

    # get latest usage per account (via customers)
    usage_logs_df["activity_date"] = pd.to_datetime(usage_logs_df["activity_date"])

    # map customer → account
    customer_account_map = usage_logs_df.merge(
        customers_df[["customer_id", "account_id"]],
        on="customer_id",
        how="left"
    )

    # recent usage (last 60 days)
    cutoff_date = datetime.today() - timedelta(days=60)
    recent_usage = customer_account_map[
        customer_account_map["activity_date"] >= cutoff_date
    ]

    usage_counts = recent_usage.groupby("account_id").size().to_dict()

    for _, sub in subscriptions_df.iterrows():
        account_id = sub["account_id"]
        status = sub["status"]

        # condition 1: cancelled subscription
        if status == "cancelled":
            churn_events.append({
                "churn_id": churn_id,
                "account_id": account_id,
                "churn_date": sub["end_date"],
                "reason": "subscription_cancelled"
            })
            churn_id += 1

        # condition 2: low usage signal
        usage_count = usage_counts.get(account_id, 0)

        if usage_count < 5 and status == "active":
            churn_events.append({
                "churn_id": churn_id,
                "account_id": account_id,
                "churn_date": datetime.today().date(),
                "reason": "low_engagement"
            })
            churn_id += 1

    return pd.DataFrame(churn_events)

#### Main Function

In [11]:
# imports
import pandas as pd
import random
from faker import Faker
from datetime import datetime, timedelta
import json

fake = Faker()

# global settings
NUM_ACCOUNTS = 120

# main execution
def main():
    accounts = generate_accounts()
    customers = generate_customers(accounts)
    deals_stages = generate_deal_stages()
    sales_reps = generate_sales_reps()
    deals = generate_deals(accounts)
    activities = generate_activities(deals)
    subscriptions = generate_subscriptions(accounts)
    transactions = generate_transactions(subscriptions)
    usage_logs = generate_usage_logs(customers)
    churn_events = generate_churn_events(accounts, customers, subscriptions, usage_logs)

    # export to CSV
    accounts.to_csv("accounts.csv", index=False)
    customers.to_csv("customers.csv", index=False)
    deals_stages.to_csv("deal_stages.csv", index=False)
    sales_reps.to_csv("sales_reps.csv", index=False)
    deals.to_csv("deals.csv", index=False)
    activities.to_csv("activities.csv", index=False)
    subscriptions.to_csv("subscriptions.csv", index=False)
    transactions.to_csv("transactions.csv", index=False)
    usage_logs.to_csv("usage_logs.csv", index=False)
    churn_events.to_csv("churn_events.csv", index=False)

if __name__ == "__main__":
    main()